# Restricted Syftr subset: four-method batch run
Attach the private v2 dataset; GPU T4 x2; Internet ON. Select **Save Version → Save & Run All**.
This keeps notebook execution active while setup, the eight-hour search window and one-hour confirmation run.
Completion is not guaranteed; incomplete budgets are labeled paused and exported for resume.


In [ ]:
def run_single_batch():
    import fcntl
    from pathlib import Path
    guard=Path("/kaggle/working/.syftr_batch_guard.lock").open("a")
    try:
        fcntl.flock(guard,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:
        print("A batch is already active in this session. No second batch was started.",flush=True)
        guard.close()
        return
    try:
        """Restore verified logical asset contents after Kaggle recursively extracts archives."""
        from pathlib import Path
        import hashlib
        import json
        import shutil
        import tarfile
        import tempfile
        import zipfile


        def file_sha(path):
            h=hashlib.sha256()
            with Path(path).open('rb') as f:
                for chunk in iter(lambda:f.read(8*1024*1024),b''):
                    h.update(chunk)
            return h.hexdigest()


        def safe_name(name):
            p=Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Unsafe package path: '+name)


        def find_expanded_asset(inputs, asset):
            anchor=asset['anchor']
            failures=[]
            for located in inputs.rglob(Path(anchor).name):
                root=located
                for _ in Path(anchor).parts:
                    root=root.parent
                if not located.is_file() or file_sha(located)!=asset['files'][anchor]['sha256']:
                    continue
                missing=None
                for rel,item in asset['files'].items():
                    p=root/rel
                    if not p.is_file():
                        if item['bytes']==0 and item['sha256']==hashlib.sha256(b'').hexdigest():
                            continue
                        missing=rel;break
                    if p.is_symlink() or p.stat().st_size!=item['bytes'] or file_sha(p)!=item['sha256']:
                        missing=rel;break
                if missing is None:
                    return root
                failures.append(str(root)+': '+missing)
            raise FileNotFoundError('Expanded asset did not match its original per-file hashes: '+asset['name']+'; '+str(failures[:3]))


        def prepare_verified_package(inputs, working, archive_name, expected_archive, expected_manifest, assets):
            inputs,working=Path(inputs),Path(working)
            working.mkdir(parents=True,exist_ok=True)
            stage=Path(tempfile.mkdtemp(prefix='syftr-expanded-fix-',dir=working))
            archives=list(inputs.rglob(archive_name))
            retained=[p for p in archives if p.is_file() and file_sha(p)==expected_archive]
            if retained:
                with zipfile.ZipFile(retained[0]) as z:
                    for name in z.namelist():safe_name(name)
                    z.extractall(stage)
                source=stage
            else:
                matches=[p for p in inputs.rglob('MANIFEST.json') if file_sha(p)==expected_manifest]
                if len(matches)!=1:
                    raise ValueError('Attach exactly the matching private dataset for this notebook; manifest not found or ambiguous')
                source=matches[0].parent
            original=source/'MANIFEST.json'
            if file_sha(original)!=expected_manifest:
                raise ValueError('Original package manifest changed')
            manifest=json.loads(original.read_text())
            derived=dict(manifest)
            reconstructed={}
            for rel,checksum in manifest.items():
                safe_name(rel)
                target=stage/rel
                p=source/rel
                if p.is_file():
                    if file_sha(p)!=checksum:
                        raise ValueError('Original file checksum mismatch: '+rel)
                    if p!=target:
                        target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,target)
                    continue
                if rel not in assets:
                    raise FileNotFoundError('Required package file is missing: '+str(p))
                asset=assets[rel]
                root=find_expanded_asset(inputs,asset)
                empty=stage/('_empty_'+str(len(reconstructed)))
                empty.write_bytes(b'')
                def source_file(name):
                    candidate=root/name
                    return candidate if candidate.is_file() else empty
                if asset['format']=='zip':
                    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
                        for name in asset['files']:
                            safe_name(name);z.write(source_file(name),name)
                elif asset['format']=='tar.gz':
                    with tarfile.open(target,'w:gz',compresslevel=1,dereference=True) as t:
                        for name in asset['files']:
                            safe_name(name);t.add(source_file(name),arcname=name,recursive=False)
                else:
                    raise ValueError('Unknown archive format')
                empty.unlink()
                derived[rel]=file_sha(target)
                reconstructed[rel]={'original_container_sha256':checksum,'repacked_container_sha256':derived[rel],
                    'verified_logical_files':len(asset['files']),'expanded_root':str(root),
                    'rule':'All original per-file bytes verified; only archive container metadata/encoding reconstructed.'}
            original_bytes=original.read_bytes()
            (stage/'ORIGINAL_MANIFEST.json').write_bytes(original_bytes)
            (stage/'MANIFEST.json').write_text(json.dumps(derived,indent=2)+'\n')
            (stage/'ASSET_RECONSTRUCTION.json').write_text(json.dumps({'original_manifest_sha256':expected_manifest,
                'assets':reconstructed,'original_attached_files_modified':False},indent=2)+'\n')
            for rel,checksum in derived.items():
                if file_sha(stage/rel)!=checksum:raise ValueError('Derived working package failed verification: '+rel)
            print('VERIFIED PACKAGE READY. Rebuilt nested archives:',list(reconstructed),flush=True)
            return stage

        # Trusted inventory generated from the original release, not the expanded dataset.
        import base64,zlib,sys,subprocess
        ASSETS=json.loads(zlib.decompress(base64.b64decode('...')))
        staging=prepare_verified_package('/kaggle/input','/kaggle/working','syftr_restricted_v2.zip','03672d8b049fa20aa5818b970faef791fe31ef8b9f666e021fdcd7a455eabfb3','f6424a0fc3f8c0558f8d026abcf3809b9bf8917715408cce228c4f50bf7b9891',ASSETS)
        # Preserve reconstruction evidence with the installed campaign records.
        status_file=Path('/kaggle/working/v2_setup_status.json')
        if not status_file.exists(): status_file.write_text(json.dumps({'status':'starting_setup'}))
        SETUP_TEST_ISOLATION_LAUNCHER='"""Setup-only test isolation overlay; research source files remain byte-identical."""\nimport hashlib\nimport fcntl\nimport os\nimport urllib.request\nimport json\nfrom pathlib import Path\nimport runpy\nimport subprocess\nimport sys\nimport time\n\nTEST_RUNNER = r\'\'\'\nimport sys,tempfile,unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\nfrom ourwork.campaign_v2 import notebook,test_campaign\noriginal=test_campaign.BackgroundControlTests.test_notebook_reads_controller_status_path\n\ndef isolated_status_test(self):\n    original_path=notebook.Path\n    with tempfile.TemporaryDirectory(prefix=\'syftr-status-isolation-\') as directory:\n        absent=Path(directory)/\'setup_status_not_present.json\'\n        def fixture_path(*args,**kwargs):\n            if len(args)==1 and str(args[0])==\'/kaggle/working/v2_setup_status.json\':\n                return absent\n            return original_path(*args,**kwargs)\n        with patch.object(notebook,\'Path\',side_effect=fixture_path):\n            original(self)\n\ntest_campaign.BackgroundControlTests.test_notebook_reads_controller_status_path=isolated_status_test\nunittest.main(module=None,argv=[\'unittest\']+sys.argv[1:])\n\'\'\'\n\n\ndef restricted_snapshot_overlay(text):\n    replacements={\n        "snapshot = OUTPUT / \'executed_source_gpu1\'": "snapshot = OUTPUT / \'executed_source_restricted_v2\'",\n        "\'source_directory\': \'executed_source_gpu1\'": "\'source_directory\': \'executed_source_restricted_v2\'",\n    }\n    for before,after in replacements.items():\n        if text.count(before)!=1:\n            raise ValueError(\'Unexpected installer layout; refusing an unverified snapshot-path change\')\n        text=text.replace(before,after)\n    compile(text,\'verified-restricted-installer\',\'exec\')\n    return text\n\n\ndef launch_setup(script, working=Path(\'/kaggle/working\')):\n    script=Path(script).resolve()\n    manifest=json.loads((script.parent/\'MANIFEST.json\').read_text())\n    assert hashlib.sha256(script.read_bytes()).hexdigest()==manifest[\'install_and_restore.py\']\n    working=Path(working)\n    lock_path=working/\'.v2_setup.lock\'\n    held=lock_path.open(\'a\')\n    original_flock=fcntl.flock\n    waited=False\n    while True:\n        try:\n            original_flock(held,fcntl.LOCK_EX|fcntl.LOCK_NB)\n            break\n        except BlockingIOError:\n            if not waited:\n                print(\'ATTACHED: an earlier setup owns the lock. Waiting without starting duplicate installation.\',flush=True)\n                waited=True\n            time.sleep(3)\n    try:\n        state_file=working/\'v2_setup_status.json\'\n        base=working/\'baseline_results\'\n        campaign=base/\'campaign-v2\'\n        profile={}\n        try:\n            profile=json.loads((script.parent/\'source/ourwork/campaign_v2/protocol.json\').read_text())\n            installed=json.loads((working/\'syftr_project/ourwork/campaign_v2/protocol.json\').read_text())\n            ready=(profile==installed and\n                json.loads((campaign/\'readiness/software_validation.json\').read_text()).get(\'status\')==\'passed\' and\n                json.loads((campaign/\'fresh_embedding_readiness.json\').read_text()).get(\'status\')==\'passed\')\n            if profile.get(\'space_variant\')==\'restricted-v2\':\n                cap=json.loads((base/\'capabilities-restricted-v2-seed11/analysis/summary.json\').read_text())\n                ready=ready and cap[\'completed\']==11 and cap[\'failed\']==0\n            if ready:\n                with urllib.request.urlopen(\'http://127.0.0.1:8765/health\',timeout=5) as response:\n                    ready=response.status==200\n            if ready:\n                state_file.write_text(json.dumps({\'status\':\'ready\',\'note\':\'Existing validated setup reused; duplicate status repaired.\'})+\'\\n\')\n                print(\'EXISTING SETUP READY: reusing validated environment and model server.\',flush=True)\n                return\n        except (OSError,ValueError,KeyError):\n            pass\n        # This launcher already holds the exact lock. Avoid acquiring the same\n        # file through a second open-file description inside the unchanged installer.\n        def coordinated_flock(fd,operation):\n            try:\n                path=os.readlink(f\'/proc/self/fd/{fd if isinstance(fd,int) else fd.fileno()}\')\n            except OSError:\n                path=\'\'\n            if path==str(lock_path) and operation & fcntl.LOCK_EX:\n                return None\n            return original_flock(fd,operation)\n        fcntl.flock=coordinated_flock\n        run_original=subprocess.run\n        def run_with_test_isolation(command,*args,**kwargs):\n            if (isinstance(command,(list,tuple)) and len(command)>3 and list(command[1:3])==[\'-m\',\'unittest\']\n                    and \'ourwork.campaign_v2.test_campaign\' in command):\n                command=[command[0],\'-c\',TEST_RUNNER,*command[3:]]\n            return run_original(command,*args,**kwargs)\n        subprocess.run=run_with_test_isolation\n        evidence=working/\'baseline_results/campaign-v2/readiness\'\n        evidence.mkdir(parents=True,exist_ok=True)\n        (evidence/f\'test_isolation_overlay_{time.time_ns()}.json\').write_text(json.dumps({\n            \'target\':\'BackgroundControlTests.test_notebook_reads_controller_status_path\',\n            \'change\':\'Only this fixture sees a nonexistent setup-status path; every test/assertion still runs.\',\n            \'setup_lock\':\'Exclusively held by the setup launcher for the entire installer execution.\',\n            \'research_source_modified\':False,\'installer_source_modified\':False,\n            \'verified_installer_sha256\':manifest[\'install_and_restore.py\'],\n            \'test_overlay_sha256\':hashlib.sha256(TEST_RUNNER.encode()).hexdigest()},indent=2)+\'\\n\')\n        sys.argv=[str(script)]\n        if profile.get(\'space_variant\')==\'restricted-v2\':\n            original_text=script.read_text()\n            derived_text=restricted_snapshot_overlay(original_text)\n            (evidence/f\'restricted_snapshot_fix_{time.time_ns()}.json\').write_text(json.dumps({\n                \'original_installer_sha256\':manifest[\'install_and_restore.py\'],\n                \'derived_execution_sha256\':hashlib.sha256(derived_text.encode()).hexdigest(),\n                \'change\':\'Restricted source archived separately in executed_source_restricted_v2.\',\n                \'original_full_snapshot_preserved\':True,\'research_source_modified\':False},indent=2)+\'\\n\')\n            namespace={\'__name__\':\'__main__\',\'__file__\':str(script),\'__package__\':None}\n            exec(compile(derived_text,str(script),\'exec\'),namespace)\n        else:\n            runpy.run_path(str(script),run_name=\'__main__\')\n    finally:\n        if \'run_original\' in locals():\n            subprocess.run=run_original\n        fcntl.flock=original_flock\n        held.close()\n\n\nif __name__==\'__main__\':\n    launch_setup(sys.argv[1])\n'
        with Path('/kaggle/working/v2_setup.log').open('a') as log:
            setup=subprocess.Popen([sys.executable,'-u','-c',SETUP_TEST_ISOLATION_LAUNCHER,str(staging/'install_and_restore.py')],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        sys.path.insert(0,str(staging/'source'))
        from ourwork.campaign_v2.notebook import status,launch,stop,downloads,confirm
        print('Verified setup started:',setup.pid,flush=True)

        # This continuation is for Kaggle Save Version -> Save & Run All, not manual Run All.
        import time
        batch_started=time.monotonic()
        def wait_with_compact_progress(process):
            while True:
                try:
                    return process.wait(timeout=60)
                except subprocess.TimeoutExpired:
                    path=Path('/kaggle/working/baseline_results/campaign-v2/status.json')
                    if path.exists():
                        record=json.loads(path.read_text())
                        p=record.get('progress',{})
                        print({k:record.get(k) for k in ['status','active_method','planned_checkpoint','latest_matched_checkpoint']},
                              {k:p.get(k) for k in ['attempts','completed','failed']},flush=True)
                    else:
                        path=Path('/kaggle/working/v2_setup_status.json')
                        print('SETUP',path.read_text() if path.exists() else 'starting',flush=True)
        code=wait_with_compact_progress(setup)
        assert code==0, 'Setup failed. Inspect v2_setup.log; no search budget should be started.'
        research_python='/kaggle/temp/rm-syftr/venv/bin/python'
        project='/kaggle/working/syftr_project'
        driver_log=Path('/kaggle/working/baseline_results/campaign-v2/controller.log').open('a')
        from ourwork.campaign_v2.notebook import active_experiments
        while active_experiments():
            print("Existing experiment is active; waiting without starting a duplicate.",flush=True)
            time.sleep(15)
        search=subprocess.Popen([research_python,'-u','-m','ourwork.campaign_v2.controller','--hours','8'],cwd=project,stdout=driver_log,stderr=subprocess.STDOUT)
        try:
            assert wait_with_compact_progress(search)==0, 'Campaign controller failed; inspect controller.log and capsule.'
            state=json.loads(Path('/kaggle/working/baseline_results/campaign-v2/status.json').read_text())
            if state['status']=='search_complete':
                with Path('/kaggle/working/baseline_results/campaign-v2/confirmation.log').open('a') as held_log:
                    check=subprocess.Popen([research_python,'-u','-m','ourwork.campaign_v2.confirm','--hours','1'],cwd=project,stdout=held_log,stderr=subprocess.STDOUT)
                    assert wait_with_compact_progress(check)==0, 'Confirmation failed; inspect confirmation.log.'
            else:
                print('SEARCH PAUSED: declared budgets remain incomplete. Resume from the exported capsule.',flush=True)
        finally:
            driver_log.close()
            subprocess.run([research_python,'-c','from ourwork.campaign_v2.snapshots import snapshot; print(snapshot())'],cwd=project,check=True)
            print('Download campaign_v2_exports/campaign_v2_review.zip and campaign_v2_receipt.json from version Output.',flush=True)

    finally:
        guard.close()

run_single_batch()
